## Setup and Imports

We start by importing the necessary libraries. This includes **Pandas** for data manipulation, components from **scikit-learn** for preprocessing, pipeline construction, data splitting, and cross-validation, alongside our baseline classification models.

In [151]:
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

## Data Loading and Specific Handling

We load our processed Titanic dataset and select our feature subset. To address missing values in the `Age` column without introducing broad bias, we perform a **stratified median imputation** grouped by passenger class (`Pclass`) and gender (`Sex`).

In [152]:
data = pd.read_csv("data/titanic_processed.csv")

columns = [
    "Sex",
    "Pclass",
    "Fare",
    "FamilySize",
    "TicketGroupSize",
    "SibSpGroup",
    "TicketPrefixGroup",
    "ParchGroup",
    "Age",
    "Survived",
]
data['Age'] = data.groupby(['Pclass', 'Sex'])['Age'].transform(lambda x: x.fillna(x.median()))
final_df = data[columns].copy()
print(final_df)

        Sex  Pclass     Fare   FamilySize TicketGroupSize SibSpGroup  \
0      male       3   7.2500  smallFamily               1        1-2   
1    female       1  71.2833  smallFamily               1        1-2   
2    female       3   7.9250        alone               1          0   
3    female       1  53.1000  smallFamily               2        1-2   
4      male       3   8.0500        alone               1          0   
..      ...     ...      ...          ...             ...        ...   
886    male       2  13.0000        alone               1          0   
887  female       1  30.0000        alone               1          0   
888  female       3  23.4500  smallFamily               2        1-2   
889    male       1  30.0000        alone               1          0   
890    male       3   7.7500        alone               1          0   

    TicketPrefixGroup ParchGroup   Age  Survived  
0                   A          0  22.0         0  
1                   P          0 

## Categorical Encoding & Dataset Partitioning

Before fitting our models, we convert our categorical features into numerical representations using **One-Hot Encoding** (`pd.get_dummies`). We then split the data into an 80/20 train-test split, utilizing **stratified sampling** on our target variable (`Survived`) to ensure identical class distributions across both sets.

In [153]:
X, Y  = final_df[columns[0:-1]].copy(),final_df["Survived"]

columns_to_encode = ['Sex', 'FamilySize', 'SibSpGroup', 'TicketGroupSize', 'TicketPrefixGroup', 'ParchGroup']
X = pd.get_dummies(X, columns=columns_to_encode, drop_first=True)

train_X, test_X, train_Y, test_Y = train_test_split(X, Y, test_size=0.2, random_state=42, stratify=Y)

## Hyperparameter Optimization Spaces

We define our modeling suite using a configurations dictionary. For each classifier (Decision Tree, Random Forest, Gradient Boosting, and Logistic Regression), we declare the underlying estimator and a specific **hyperparameter grid** to test during tuning. Note that Logistic Regression utilizes a pipeline to ensure proper feature scaling.

In [154]:
configs = {
    "Decision Tree": {
        "model": DecisionTreeClassifier(random_state=42),
        "grid": {
            "criterion": ["gini", "entropy"],
            "max_depth": [3, 5, 7, 10, None],
            "min_samples_split": [2, 5, 10],
            "min_samples_leaf": [1, 2, 4]
        }
    },
    "Random Forest": {
        "model": RandomForestClassifier(random_state=42),
        "grid": {
            "n_estimators": [100, 200, 300],
            "criterion": ["gini", "entropy"],
            "max_depth": [4, 6, 8, 10],
            "min_samples_split": [2, 5, 10]
        }
    },
    "Gradient Boosting": {
        "model": GradientBoostingClassifier(random_state=42),
        "grid": {
            "n_estimators": [50, 100, 150],
            "learning_rate": [0.01, 0.05, 0.1, 0.2],
            "max_depth": [3, 4, 5],
            "subsample": [0.8, 1.0]
        }
    },
    "Logistic Regression": {
        "model": Pipeline([
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(max_iter=1000, random_state=42))
        ]),
        "grid": {
            "model__C": [0.01, 0.1, 1, 10, 100],
            "model__solver": ["liblinear", "lbfgs"]
        }
    }
}

## Grid Search Cross-Validation

We loop through our defined models and execute a **5-fold Grid Search Cross-Validation** (`GridSearchCV`) using all available CPU cores (`n_jobs=-1`). This allows us to find the parameter configurations that maximize out-of-fold validation accuracy, protecting against overfitting.

In [155]:
best_models = {}
cv_scores = {}

for name, config in configs.items():

    grid_search = GridSearchCV(
        estimator=config["model"],
        param_grid=config["grid"],
        cv=5,
        scoring="accuracy",
        n_jobs=-1
    )

    grid_search.fit(train_X, train_Y)

    best_model = grid_search.best_estimator_
    best_models[name] = best_model

    test_score = best_model.score(test_X, test_Y)
    cv_scores[name] = grid_search.best_score_

    print(f"  Optimal Params: {grid_search.best_params_}")
    print(f"  Train CV Accuracy: {cv_scores[name]:.4f}")
    print(f"  Test Accuracy:    {test_score:.4f}\n")

C:\Users\User\AppData\Local\Programs\Python\Python314\Lib\site-packages\joblib\externals\loky\process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


  Optimal Params: {'criterion': 'gini', 'max_depth': 10, 'min_samples_leaf': 1, 'min_samples_split': 10}
  Train CV Accuracy: 0.8231
  Test Accuracy:    0.7933

  Optimal Params: {'criterion': 'gini', 'max_depth': 10, 'min_samples_split': 10, 'n_estimators': 300}
  Train CV Accuracy: 0.8358
  Test Accuracy:    0.7877

  Optimal Params: {'learning_rate': 0.2, 'max_depth': 3, 'n_estimators': 50, 'subsample': 0.8}
  Train CV Accuracy: 0.8316
  Test Accuracy:    0.8045

  Optimal Params: {'model__C': 0.1, 'model__solver': 'lbfgs'}
  Train CV Accuracy: 0.8119
  Test Accuracy:    0.7989



## Performance Evaluation

With our optimal models secured, we extract their final architectures and evaluate their generalization capabilities by comparing their cross-validation performance against their final accuracy on the unseen test partition.

In [156]:
for model in best_models.values():
    print(model)

DecisionTreeClassifier(max_depth=10, min_samples_split=10, random_state=42)
RandomForestClassifier(max_depth=10, min_samples_split=10, n_estimators=300,
                       random_state=42)
GradientBoostingClassifier(learning_rate=0.2, n_estimators=50, random_state=42,
                           subsample=0.8)
Pipeline(steps=[('scaler', StandardScaler()),
                ('model',
                 LogisticRegression(C=0.1, max_iter=1000, random_state=42))])


In [157]:
for name, model in best_models.items():
    test_score = model.score(test_X, test_Y)

    cv_score = cv_scores[name]

    print(f"{name}")
    print(f"  Test Accuracy: {test_score:.4f}")
    print(f"  CV Accuracy:   {cv_score:.4f}")

Decision Tree
  Test Accuracy: 0.7933
  CV Accuracy:   0.8231
Random Forest
  Test Accuracy: 0.7877
  CV Accuracy:   0.8358
Gradient Boosting
  Test Accuracy: 0.8045
  CV Accuracy:   0.8316
Logistic Regression
  Test Accuracy: 0.7989
  CV Accuracy:   0.8119


## Ensemble Modeling (Soft Voting)

To maximize performance, we combine our top performing individual estimators (Logistic Regression, Random Forest, and Gradient Boosting) into a **Soft Voting Classifier**. Instead of relying on a single model's hard prediction, this ensemble averages the predicted probabilities across all three architectures, typically leading to more robust decisions and a higher test score.

In [158]:
from sklearn.ensemble import VotingClassifier

voting_model = VotingClassifier(
    estimators=[('lr', best_models['Logistic Regression']), ('rf', best_models['Random Forest']), ('gb', best_models['Gradient Boosting'])],
    voting='soft'
)
voting_model.fit(train_X, train_Y)
print("Ensemble Test Accuracy:", voting_model.score(test_X, test_Y))

Ensemble Test Accuracy: 0.8212290502793296
